In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27


def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })


def fig_textwidth(height_ratio=0.62):
    return plt.subplots(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio))


# call once in your notebook/script
setup_pub_style(fontsize=9)
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette("colorblind")


def plot_cross_design(results_dict, metric="val_loss", cmap="magma_r", save_path=None):
    """
    Plot cross-design grid of best validation loss using publication-style formatting.

    Parameters
    ----------
    results_dict : dict
        {(depth, width): csv_path}

        Example:
        {
            (4, 8): "runs/d4_w8/loss.csv",
            (4, 16): "runs/d4_w16/loss.csv",
            (4, 32): "runs/d4_w32/loss.csv",
            (3, 16): "runs/d3_w16/loss.csv",
            (5, 16): "runs/d5_w16/loss.csv",
        }

    metric : str
        Column to minimize ("val_loss" default)

    cmap : str
        Matplotlib colormap. Default "cividis_r" is colorblind-friendly and
        reversed so lower loss appears darker/stronger.
    """

    depths = sorted({k[0] for k in results_dict})
    widths = sorted({k[1] for k in results_dict})

    grid = np.full((len(depths), len(widths)), np.nan)

    for (depth, width), path in results_dict.items():
        df = pd.read_csv(path)
        best = df[metric].min()

        i = depths.index(depth)
        j = widths.index(width)

        grid[i, j] = best

    fig, ax = fig_textwidth(height_ratio=0.5)

    masked = np.ma.masked_invalid(grid)
    im = ax.imshow(masked, cmap=cmap, aspect="equal")

    # blank missing cells
    im.cmap.set_bad(color="white")

    ax.set_xticks(range(len(widths)))
    ax.set_xticklabels(widths)
    ax.set_yticks(range(len(depths)))
    ax.set_yticklabels(depths)

    ax.set_xlabel("Base channels (width)")
    ax.set_ylabel("Model layers (depth)")
    # ax.set_title("Best validation loss")

    # draw table-like cell borders
    ax.set_xticks(np.arange(-0.5, len(widths), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(depths), 1), minor=True)
    ax.grid(which="minor", color="black", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)

    # remove major grid from seaborn whitegrid
    ax.grid(False)

    # annotate values
    center_i = len(depths) // 2
    center_j = len(widths) // 2

    for i in range(len(depths)):
        for j in range(len(widths)):
            if np.isfinite(grid[i, j]):
                txt_color = "black" if (i == center_i and j == center_j) else "white"
                font = "bold" if (i == center_i and j == center_j) else "normal"

                ax.text(
                    j, i, f"{grid[i, j]:.4f}",
                    ha="center", va="center",
                    color=txt_color,
                    fontweight=font
                )


    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Validation loss")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


In [ ]:
local_path = ""

In [ ]:
runs = {
    (3,8):  f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_3layers_drift_wFutureWind_SAR_all_base8_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (3,32): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_3layers_drift_wFutureWind_SAR_all_base32_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (4,8):  f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_4layers_drift_wFutureWind_SAR_all_base8_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (4,16): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_4layers_drift_wFutureWind_SAR_all_base16_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (4,32): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_4layers_drift_wFutureWind_SAR_all_base32_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (3,16): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_3layers_drift_wFutureWind_SAR_all_base16_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (5,16): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_5layers_drift_wFutureWind_SAR_all_base16_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (5,8):  f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_5layers_drift_wFutureWind_SAR_all_base8_lr3e-4_bs16_wDivLoss_masked/loss.csv",
    (5,32): f"{local_path}model_dev_main/runs/min400PM_experiments/DivLoss/unet_5layers_drift_wFutureWind_SAR_all_base32_lr3e-4_bs16_wDivLoss_masked/loss.csv",
}

plot_cross_design(runs, save_path="depth_width_grid_search_magma.pdf")
